# GTEx histology: H&E whole-slide images of healthy human tissues

The Genotype-Tissue Expression (GTEx) project sampled healthy tissues across the body from each of its donors. This store holds the project's H&E-stained slides, 25,503 of them from 971 donors and 40 tissue types as IDC codes them, as NCI Imaging Data Commons (IDC) publishes them: each slide at full resolution (about 0.5 µm per pixel, scanned at 20x) and at the scanner's coarser levels, with a thumbnail, a photograph of the glass slide and the slide's DICOM headers. The slides are CC BY 4.0; the store's root lists the works to cite.

This notebook opens the store with nothing else: no copy of the slides, no DICOM reader.

**Requirements:** `zarr`, `xarray`, `icechunk` and `imagecodecs`, plus `arraylake` to open the published repo. imagecodecs decodes the slides' JPEG tiles; the store names it in its root attributes `codecs` and `reading`. `STORE` is the published repo, or a local build's path.

In [ ]:
%pip install --quiet imagecodecs

In [ ]:
import json
import os
from collections import Counter

import icechunk
import imagecodecs
import imagecodecs.zarr
import numpy as np
import xarray as xr
import zarr
from IPython.display import Image, display

# Registers imagecodecs' Zarr codecs, among them imagecodecs_jpeg, which the slide tiles name.
imagecodecs.zarr.register_codecs()

STORE = os.environ.get("IDC_GTEX_STORE", "arraylake://earthmover-biology/idc-gtex")

if STORE.startswith("arraylake://"):
    # The published repo lives in Arraylake, whose client applies its virtual chunk containers.
    from arraylake import Client

    repo = Client().get_repo(STORE.removeprefix("arraylake://"))
else:
    # The slides' bytes stay in IDC's bucket, which anyone may read without credentials.
    access = {"s3://idc-open-data/": icechunk.Credentials.S3(icechunk.s3_anonymous_credentials())}
    repo = icechunk.Repository.open(icechunk.local_filesystem_storage(STORE), authorize_virtual_chunk_access=access)
session = repo.readonly_session("main")
root = zarr.open_group(session.store, mode="r")

print(root.attrs["title"])
print(root.attrs["description"])
print()
print("license:", root.attrs["license"], "· IDC release:", root.attrs["idc_version"])
print("codecs a reader needs:", root.attrs["codecs"])
print(root.attrs["reading"])
print("slides left out:", len(root.attrs["left_out"]))

## Donors and their slides

The hierarchy is `gtex/<donor>/<slide>`, named as IDC names them: a donor is a GTEx subject ID, a slide is the barcode on its glass. Each donor and slide group describes itself, and each slide says what tissue it shows.

In [ ]:
gtex = root["gtex"]
print(gtex.attrs["description"])

slides = {path: group for path, group in gtex.members(max_depth=1) if path.count("/") == 1}
donors = {path.split("/")[0] for path in slides}
print(f"{len(donors)} donors, {len(slides)} slides")

tissues = Counter(group.attrs["specimen"]["tissue"] for group in slides.values())
for tissue, n in tissues.most_common(8):
    print(f"{n:6d}  {tissue}")

first_donor = sorted(donors)[0]
print()
print(gtex[first_donor].attrs["description"])

## One slide

A slide's `image` is the scanner's pyramid: level 0 at full resolution, each further level a coarser copy. The `scale` says how big a pixel is, in micrometers, and the `translation` where its first pixel's center sits. Arrays are read in 240 x 240 pixel tiles, the tiles the scanner wrote.

In [ ]:
def transforms(group):
    """Each level's (scale, translation), from an OME-Zarr 0.6 image group's metadata."""
    out = {}
    for dataset in group.attrs["ome"]["multiscales"][0]["datasets"]:
        steps = {t["type"]: t[t["type"]] for t in dataset["coordinateTransformations"][0]["transformations"]}
        out[dataset["path"]] = (steps["scale"], steps["translation"])
    return out


path, slide = sorted(slides.items())[0]
print(slide.attrs["description"])

image = slide["image"]
print(image.attrs["description"])
levels = {}
for name, (scale, translation) in transforms(image).items():
    level = image[name]
    levels[name] = (level, scale, translation)
    print(f"level {name}: {level.shape}, {scale[1]:.3f} µm pixels")

## Real pixels, in micrometers

`region` reads a window of one level as an `xarray.DataArray` whose `y` and `x` coordinates are pixel centers in micrometers, from the level's scale and translation. `show` encodes it as PNG with imagecodecs, so no plotting library is needed.

Much of a slide is bare glass, so the window is centered where the thumbnail is darkest on average over a 0.5 mm square: the densest tissue. The same 0.5 mm square is read at level 0, about 1,000 pixels across, and again at a coarser level.

In [ ]:
def region(level_path, y_um, x_um):
    """The window of one level covering y_um and x_um, each (start, stop) in micrometers."""
    level, scale, translation = levels[level_path]
    (y0, y1), (x0, x1) = (
        (int((start - translation[axis]) / scale[axis]), int((stop - translation[axis]) / scale[axis]) + 1)
        for axis, (start, stop) in zip((1, 2), (y_um, x_um))
    )
    pixels = level[:, y0:y1, x0:x1]
    return xr.DataArray(
        pixels,
        dims=("c", "y", "x"),
        coords={
            "y": translation[1] + scale[1] * np.arange(y0, y0 + pixels.shape[1]),
            "x": translation[2] + scale[2] * np.arange(x0, x0 + pixels.shape[2]),
        },
        attrs={"units": "micrometer"},
    )


def show(rgb, width=480):
    """An RGB DataArray as a PNG, shown ``width`` screen pixels wide."""
    # png_encode needs the (y, x, c) pixels in one contiguous block.
    pixels = np.ascontiguousarray(rgb.transpose("y", "x", "c").values)
    display(Image(data=imagecodecs.png_encode(pixels), width=width))


# The thumbnail's darkest 0.5 mm square, found with a box sum over its darkness.
thumbnail = slide["thumbnail"]
(thumb_scale, thumb_translation), = transforms(thumbnail).values()
darkness = 255 - thumbnail["0"][:].mean(axis=0)
k = max(1, round(500 / thumb_scale[1]))
box = np.pad(darkness, ((1, 0), (1, 0))).cumsum(0).cumsum(1)
sums = box[k:, k:] - box[:-k, k:] - box[k:, :-k] + box[:-k, :-k]
ty, tx = np.unravel_index(sums.argmax(), sums.shape)
cy, cx = ((t + k / 2) * thumb_scale[i] for t, i in ((ty, 1), (tx, 2)))
square = ((cy - 250, cy + 250), (cx - 250, cx + 250))

for name in ("0", str(len(levels) // 2)):
    rgb = region(name, *square)
    print(f"level {name}: {rgb.sizes['y']} x {rgb.sizes['x']} pixels, "
          f"y {float(rgb.y[0]):,.0f} to {float(rgb.y[-1]):,.0f} µm, x {float(rgb.x[0]):,.0f} to {float(rgb.x[-1]):,.0f} µm")
    show(rgb)

## The whole slide

The thumbnail shows the whole slide and decodes with `zarr` alone. Its scale and translation put it in the same micrometer frame as the image, so `xarray` labels both axes in micrometers.

In [ ]:
print(thumbnail.attrs["description"])
pixels = thumbnail["0"][:]
whole = xr.DataArray(
    pixels,
    dims=("c", "y", "x"),
    coords={axis: thumb_translation[i] + thumb_scale[i] * np.arange(pixels.shape[i]) for axis, i in (("y", 1), ("x", 2))},
    attrs={"units": "micrometer"},
)
print(whole.sizes, f"spanning {float(whole.y[-1]) / 1000:.1f} mm by {float(whole.x[-1]) / 1000:.1f} mm")
show(whole)

## What the DICOM said

Every image's DICOM header travels with the slide, in `dicom`: the scanner, the specimen and its preparation, the optical path and the pixel geometry, as one JSON document keyed by the node that holds the image.

In [ ]:
headers = json.loads(bytes(slide["dicom"][:]))
level0 = headers["image/0"]
for tag, name in [("00080070", "Manufacturer"), ("00081090", "ManufacturerModelName"),
                  ("00480006", "TotalPixelMatrixColumns"), ("00480007", "TotalPixelMatrixRows"),
                  ("00280008", "NumberOfFrames"), ("00480001", "ImagedVolumeWidth")]:
    print(f"{name:24s} {level0.get(tag, {}).get('Value')}")
print("instances:", sorted(headers))